# Traveling salesman problem

Find the closed tour that visits all ten cities and has the minimum total length. Distances between pairs of cities are given in the table below:

|         |  ATL  |  ORD  |  DEN  |  IAH  |  LAX  |  MIA  |  JFK  |  SFO  |  SEA  |  DCA  |
|--------:|------:|------:|------:|------:|------:|------:|------:|------:|------:|------:|
|**ATL**  |    0  |  587  | 1212  |  701  | 1936  |  604  |  748  | 2139  | 2182  |  543  |
|**ORD**  |  587  |    0  |  920  |  940  | 1745  | 1188  |  713  | 1858  | 1737  |  597  |
|**DEN**  | 1212  |  920  |    0  |  879  |  831  | 1726  | 1631  |  949  | 1021  | 1494  |
|**IAH**  |  701  |  940  |  879  |    0  | 1379  |  968  | 1420  | 1645  | 1891  | 1220  |
|**LAX**  | 1936  | 1745  |  831  | 1379  |    0  | 2339  | 2451  |  347  |  959  | 2300  |
|**MIA**  |  604  | 1188  | 1726  |  968  | 2339  |    0  | 1092  | 2594  | 2734  |  923  |
|**JFK**  |  748  |  713  | 1631  | 1420  | 2451  | 1092  |    0  | 2571  | 2408  |  205  |
|**SFO**  | 2139  | 1858  |  949  | 1645  |  347  | 2594  | 2571  |    0  |  678  | 2442  |
|**SEA**  | 2182  | 1737  | 1021  | 1891  |  959  | 2734  | 2408  |  678  |    0  | 2329  |
|**DCA**  |  543  |  597  | 1494  | 1220  | 2300  |  923  |  205  | 2442  | 2329  |    0  |

Open the course repository root in VS Code, select the Julia 1.12 kernel with the course environment, and choose **Run All**. All required packages are included; no external data files are needed.

In [ ]:
# Define the problem data (cities and costs)
# Run the notebook in order.
using JuMP, NamedArrays, HiGHS

cities = [:ATL, :ORD, :DEN, :IAH, :LAX, :MIA, :JFK, :SFO, :SEA, :DCA]

distances = [     0   587  1212   701  1936   604   748  2139  2182   543
                587     0   920   940  1745  1188   713  1858  1737   597
               1212   920     0   879   831  1726  1631   949  1021  1494
                701   940   879     0  1379   968  1420  1645  1891  1220
               1936  1745   831  1379     0  2339  2451   347   959  2300
                604  1188  1726   968  2339     0  1092  2594  2734   923
                748   713  1631  1420  2451  1092     0  2571  2408   205
               2139  1858   949  1645   347  2594  2571     0   678  2442
               2182  1737  1021  1891   959  2734  2408   678     0  2329
                543   597  1494  1220  2300   923   205  2442  2329     0  ]

c = NamedArray(distances,(cities,cities))
N = size(distances,1);

In [ ]:
function printTspSol(x)
    for i in 1:N
        for j in 1:N
            if x[i,j] > 0.5
                println(cities[i], " -> ", cities[j])
            end
        end
    end
end

## Assignment relaxation: a cycle cover

One incoming arc and one outgoing arc at each city permit several disjoint
cycles. This assignment LP has an integral optimal extreme point, but its
cycle cover need not be a single tour. Its objective is a lower bound on
the TSP optimum. The next two formulations enforce a single tour.

In [ ]:
# solve the simplest version (it's an assignment problem)


m = Model(HiGHS.Optimizer)
set_silent(m)

# LP relaxation
@variable(m, 0 <= x[cities,cities] <= 1)
# exactly one edge into each node
@constraint(m, c1[j in cities], sum( x[i,j] for i in cities ) == 1)
# exactly one edge out of each node
@constraint(m, c2[i in cities], sum( x[i,j] for j in cities ) == 1)
# no self-loops
@constraint(m, c3[i in cities], x[i,i] == 0 )
# minimize total cost
@objective(m, Min, sum( x[i,j]*c[i,j] for i in cities, j in cities ))
optimize!(m)
if !is_solved_and_feasible(m; allow_local = false)
    error("No optimal primal solution: " *
        "$(termination_status(m)), $(primal_status(m))")
end

# Check integrality before rounding for display.
xx = value.(x)
@assert all(isapprox(v, round(v); atol=1e-5) for v in xx)
sol = NamedArray(zeros(Int,N,N),(cities,cities))
for i in cities
    for j in cities
        sol[i,j] = round(Int, xx[i,j])
    end
end

printTspSol(sol)

## Solve TSP using adaptive subtour elimination

Solve a binary cycle-cover model, identify its cycles, and forbid each
proper cycle encountered. Re-solve until a single cycle visits every city.
The distances are symmetric, so the code forbids each encountered cycle
in both directions. These constraints exclude the specific directed cycles;
they do not enumerate all subsets in advance.

Each intermediate objective is a lower bound on the TSP optimum. Once an
optimal solve gives one complete tour, that tour is optimal for the TSP.

### Helper functions

In [ ]:
# Follow outgoing arcs and return the cycle containing START.
function getSubtour(x, start)
    subtour = [start]
    for _ in 1:N
        j = subtour[end]
        successors = [k for k in cities if x[j,k] > 0.5]
        length(successors) == 1 || error("Expected one outgoing arc at $j.")
        next_city = only(successors)
        push!(subtour, next_city)
        next_city == start && return subtour
        next_city in subtour[1:end-1] && error("Cycle did not return to its start.")
    end
    error("Cycle traversal exceeded the number of cities.")
end

# This helper requires an integer cycle cover, not a fractional LP solution.
function getAllSubtours(x)
    all(isapprox(v, round(v); atol=1e-5) for v in x) ||
        error("Expected an integer cycle cover.")
    for city in cities
        count(k -> x[city,k] > 0.5, cities) == 1 || error("Invalid out-degree.")
        count(k -> x[k,city] > 0.5, cities) == 1 || error("Invalid in-degree.")
    end
    nodesRemaining = copy(cities)
    subtours = []
    while !isempty(nodesRemaining)
        subtour = getSubtour(x, first(nodesRemaining))
        push!(subtours, subtour)
        nodesRemaining = setdiff(nodesRemaining, subtour)
    end
    return subtours
end

### Optimization problem

In [ ]:
m = Model(HiGHS.Optimizer)
set_silent(m)

# must formulate as IP this time
@variable(m, x[cities,cities], Bin)
# one in-edge
@constraint(m, c1[j in cities], sum( x[i,j] for i in cities ) == 1)
# one out-edge
@constraint(m, c2[i in cities], sum( x[i,j] for j in cities ) == 1)
# no self-loops
@constraint(m, c3[i in cities], x[i,i] == 0 )
# minimize total cost
@objective(m, Min, sum( x[i,j]*c[i,j] for i in cities, j in cities ))

sols = []

for iters = 1:30
    optimize!(m)
    if !is_solved_and_feasible(m; allow_local = false)
        error("No optimal primal solution: " *
            "$(termination_status(m)), $(primal_status(m))")
    end
    println("Tour length: ", objective_value(m))
    xx = value.(x)
    push!(sols,xx)
    # get all the subtours
    subtours = getAllSubtours(xx)
    display(subtours)
    len = length(subtours)
    # solution is just a single tour!
    if len == 1
        println("SOLVED!")
        break
    else
        iters < 30 || error("No complete tour found within 30 solves.")
        for subtour in subtours
            L = length(subtour)
            @constraint(m, sum( x[subtour[k+1],subtour[k]] for k = 1:L-1 ) <= L-2)
            @constraint(m, sum( x[subtour[k],subtour[k+1]] for k = 1:L-1 ) <= L-2)
        end
    end
end

## Miller–Tucker–Zemlin formulation

The continuous variables $u_i$ represent visit positions, with Atlanta
fixed at position 1. For an arc $i\to j$ selected with $j$ other than
Atlanta, the constraints require $u_j\ge u_i+1$. Summing these inequalities
around a cycle that excludes Atlanta would give a contradiction. Together
with the degree constraints, they force a single tour through all cities.

In [ ]:
m = Model(HiGHS.Optimizer)
set_silent(m)

# must formulate as IP this time
@variable(m, x[cities,cities], Bin)
# one in-edge
@constraint(m, c1[j in cities], sum( x[i,j] for i in cities ) == 1)
# one out-edge
@constraint(m, c2[i in cities], sum( x[i,j] for j in cities ) == 1)
# no self-loops
@constraint(m, c3[i in cities], x[i,i] == 0 )
# minimize total cost
@objective(m, Min, sum( x[i,j]*c[i,j] for i in cities, j in cities ))

# MTZ variables and constraints
@variable(m, 1 <= u[cities] <= N)
fix(u[cities[1]], 1; force=true)
@constraint(m, c4[i in cities, j in cities[2:end]], u[i] - u[j] + N*x[i,j] <= N-1 )

optimize!(m)
if !is_solved_and_feasible(m; allow_local = false)
    error("No optimal primal solution: " *
        "$(termination_status(m)), $(primal_status(m))")
end
xx = value.(x)
# get the complete tour
subtours = getAllSubtours(xx)
display(subtours)
println("Tour length: ", objective_value(m))